<a href="https://colab.research.google.com/github/jppriya95/thinkpalm-agentai-Jayapriya-Lab1/blob/main/src/CBU_TAAS_ReAct_Diagnostic_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1: Environment Setup & Diagnostic Tools
!pip install -q google-genai

import json
import time
from google.colab import userdata
from google import genai
from google.genai import types
from google.genai.errors import ServerError, ClientError

# Define Diagnostic Mock Tools
def get_test_logs() -> str:
    """Returns the test execution logs for the failing CBU TAAS test suite."""
    return """
    [2026-03-30 10:15:00] [INFO] Starting Test: Telecom Billing API Test Suite
    [2026-03-30 10:15:02] [INFO] Sending POST /api/v1/billing/charge
    [2026-03-30 10:15:32] [ERROR] API request timed out after 30 seconds
    [2026-03-30 10:15:32] [FAIL] Test Failed: Telecom Billing API Test
    """

def check_endpoint_status() -> str:
    """Checks the health and status code of the dependent Telecom Billing API endpoint."""
    return json.dumps({
        "endpoint": "https://api.telecom.internal/v1/billing",
        "status_code": 503,
        "message": "Service Unavailable - Database connection pool exhausted"
    })

tools_list = [get_test_logs, check_endpoint_status]
tool_map = {
    "get_test_logs": get_test_logs,
    "check_endpoint_status": check_endpoint_status
}

# Initialize Gemini Client
api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

In [2]:
# Cell 2: System Instructions & Retry Helper

system_instruction = """
You are an expert Telecom QA Automation Diagnostic Agent (CBU TAAS).
Your job is to diagnose test failures using available tools and reason step-by-step.

Follow the ReAct framework:
1. Thought: Reason about what to do next.
2. Action: Call an appropriate tool to gather information.
3. Observation: Analyze the tool output.
4. Final Diagnostic Report: Once you have enough information, generate a clear diagnostic report covering:
   - Test Name & Status
   - Root Cause
   - Evidence
   - Recommendation
"""

def send_chat_message_with_retry(chat, message_payload, max_retries=3):
    """Sends chat messages with exponential backoff for transient 503/429 errors."""
    for attempt in range(max_retries):
        try:
            return chat.send_message(message_payload)
        except (ServerError, ClientError) as e:
            wait_time = (2 ** attempt) * 2
            print(f"⚠️ API Warning ({e.code if hasattr(e, 'code') else 'Unavailable'}). Retrying in {wait_time}s...")
            time.sleep(wait_time)
            if attempt == max_retries - 1:
                raise e

In [3]:
# Cell 3: Execute Agent Run

def run_react_agent():
    print("=" * 60)
    print("🤖 STARTING REAL REACT AGENT FOR CBU TAAS DIAGNOSTICS")
    print("=" * 60)

    chat = client.chats.create(
        model="gemini-3.6-flash",
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            tools=tools_list,
            temperature=0.0
        )
    )

    prompt = "Diagnose why the recent CBU TAAS test suite failed."
    print(f"\n--- User / Tool Output ---\n{prompt}\n")

    current_response = send_chat_message_with_retry(chat, prompt)

    # Multi-turn tool execution loop
    for _ in range(5):
        if not current_response.function_calls:
            print(f"--- Agent ---\n{current_response.text.strip()}\n")
            break

        for call in current_response.function_calls:
            print(f"--- Agent ---")
            print(f"🔧 [ACTION] Calling Tool: {call.name} with args {dict(call.args)}\n")

            func = tool_map.get(call.name)
            tool_result = func(**call.args) if func else f"Tool {call.name} not found"

            print(f"--- User / Tool Output ---")
            print(f"👁️ [OBSERVATION]: {{'result': {repr(tool_result)}}}\n")

            current_response = send_chat_message_with_retry(
                chat,
                types.Part.from_function_response(
                    name=call.name,
                    response={"result": tool_result}
                )
            )

run_react_agent()

🤖 STARTING REAL REACT AGENT FOR CBU TAAS DIAGNOSTICS

--- User / Tool Output ---
Diagnose why the recent CBU TAAS test suite failed.

--- Agent ---
### Final Diagnostic Report

#### **Test Name & Status**
* **Test Suite:** Telecom Billing API Test Suite
* **Test Name:** Telecom Billing API Test
* **Status:** FAILED

---

#### **Root Cause**
The test suite failed because the dependent Telecom Billing API service is currently unavailable. The service backend experienced **database connection pool exhaustion**, which caused the `POST /api/v1/billing/charge` endpoint to hang and time out after 30 seconds, returning an HTTP `503 Service Unavailable` error.

---

#### **Evidence**
1. **Test Execution Logs:**
   * `[2026-03-30 10:15:02] [INFO] Sending POST /api/v1/billing/charge`
   * `[2026-03-30 10:15:32] [ERROR] API request timed out after 30 seconds`
   * `[2026-03-30 10:15:32] [FAIL] Test Failed: Telecom Billing API Test`

2. **API Endpoint Health Check:**
   * **Endpoint:** `https://api